# Special charactes in prescription text field (drug name)

In [ ]:
import pyspark
import dxpy
import hail as hl

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [ ]:
hl.init(sc=sc, default_reference='GRCh38')

#### Envinroment setup

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Input database configuration and loading

In [ ]:
db_name = 'clinical_phenos'
# full_tb_name = 'full_phenos_details_smp_0100.ht'
# full_tb_name = 'full_phenos_details_smp_0010.ht'
full_tb_name = 'full_phenos_hail_0.2.116.ht'

In [ ]:
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database", project=dxpy.PROJECT_CONTEXT_ID)['id']
url = f"dnax://{db_uri}/{full_tb_name}"
full = hl.read_table(url)

Checking dataset size.

In [ ]:
full.count()

In [ ]:
full.n_partitions()

Testing methodology.

In [ ]:
hl.eval(hl.set(hl.literal('ala maś ko;ta+').split('').filter(lambda c: ~c.matches(r'[A-Za-z0-9]'))))

Filtering out data with text field available.

In [ ]:
%time desc_df = full.filter((hl.len(full.details) > 0) & ~(hl.is_missing(full.details[0]))).repartition(24).cache()

Special characters searching.

In [ ]:
%time desc_df = desc_df.annotate(spec_chars = desc_df.details[0].split('').filter(lambda c: ~c.matches(r'[A-Za-z0-9 ]'))).cache()

## Found special characters

Aggregating charactes to result set.

In [ ]:
%time aggregated_chars = desc_df.aggregate(hl.agg.explode(lambda c: hl.agg.collect_as_set(c), desc_df.spec_chars))
aggregated_chars.remove('')
aggregated_chars

In [ ]:
%%time
import re
from datetime import datetime

source_data = desc_df.drop('spec_chars')
sample_sz = 20

rand_seed = int(datetime.now().timestamp())
source_data = source_data.annotate(rand = hl.rand_unif(0, 1, seed = rand_seed)).order_by('rand').cache()
joined = source_data.head(0).annotate(spec_char = ' ')
for char in aggregated_chars:
    re_char = hl.literal(f'([^0-9]|^){re.escape(char)}([^0-9]|$)')
    char_data = source_data.filter(source_data.details[0].matches(re_char)).head(sample_sz).annotate(spec_char = char)
    joined = joined.union(char_data)
joined = joined.cache().order_by('spec_char').drop('rand')

joined.show(-1)

In [ ]:
%%time
import re
from datetime import datetime

source_data = desc_df.drop('spec_chars')
sample_sz = 20

rand_seed = int(datetime.now().timestamp())
source_data = source_data.annotate(rand = hl.rand_unif(0, 1, seed = rand_seed)).order_by('rand').cache()
joined = source_data.head(0).annotate(spec_char = ' ')
for char in aggregated_chars:
    re_char = hl.literal(f'[0-9]{re.escape(char)}[0-9]')
    char_data = source_data.filter(source_data.details[0].matches(re_char)).head(sample_sz).annotate(spec_char = char)
    joined = joined.union(char_data)
joined = joined = joined.cache().order_by('spec_char').drop('rand')

joined.show(-1)